# Bank Account Fraud (BAF) — Exploratory Data Analysis

Dataset: [Bank Account Fraud Dataset Suite (NeurIPS 2022)](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022) — `Base.csv`

This notebook explores the dataset before model training. Key goals:
- Understand the class imbalance and temporal structure
- Analyse categorical and numerical features by fraud label
- Motivate preprocessing choices (temporal split, label encoding, scaling)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

DATA_PATH = "../DATA/Base.csv"
FIG_DIR = "../outputs/figures"
os.makedirs(FIG_DIR, exist_ok=True)

CATEGORICAL_FEATURES = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
TARGET = "fraud_bool"

## 1. Load Data & Basic Profile

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape       : {df.shape}")
print(f"Fraud count : {df[TARGET].sum():>6,}")
print(f"Legit count : {(df[TARGET]==0).sum():>6,}")
print(f"Fraud rate  : {df[TARGET].mean()*100:.3f}%")
print(f"Months      : {sorted(df['month'].unique())}")
df.head()

In [ ]:
print("Null values per column:")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "None")
df.describe()

## 2. Class Distribution

In [ ]:
counts = df[TARGET].value_counts().sort_index()
labels = ["Legitimate", "Fraud"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(labels, counts.values, color=["steelblue", "tomato"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2000, f"{v:,}", ha="center", fontsize=11)
axes[0].set_title("Transaction Counts")
axes[0].set_ylabel("Count")

axes[1].pie(
    counts.values, labels=labels, autopct="%1.2f%%",
    colors=["steelblue", "tomato"], startangle=90
)
axes[1].set_title("Class Distribution")

fig.suptitle("Class Imbalance in BAF Base Dataset", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/class_distribution.png", dpi=150)
plt.show()

## 3. Temporal Structure — Fraud Rate by Month

In [ ]:
monthly = df.groupby("month")[TARGET].agg(["sum", "count", "mean"]).reset_index()
monthly.columns = ["month", "fraud_count", "total", "fraud_rate"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(monthly["month"], monthly["total"], color="steelblue", label="Total")
axes[0].bar(monthly["month"], monthly["fraud_count"], color="tomato", label="Fraud")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Count")
axes[0].set_title("Transactions per Month")
axes[0].legend()

axes[1].plot(monthly["month"], monthly["fraud_rate"] * 100, marker="o", color="tomato")
axes[1].axvline(x=5.5, color="gray", linestyle="--", label="Train/Val boundary")
axes[1].axvline(x=6.5, color="black", linestyle="--", label="Val/Test boundary")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].set_title("Fraud Rate Over Time")
axes[1].legend()

fig.suptitle("Temporal Structure — BAF Dataset", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/temporal_structure.png", dpi=150)
plt.show()

print(monthly.to_string(index=False))

**Finding:** The dataset has temporal drift — fraud rate may vary across months. We use a temporal train/val/test split (months 1–5 / 6 / 7–8) to reflect real deployment conditions, where the model is trained on past data and evaluated on future data.

## 4. Categorical Feature Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, col in enumerate(CATEGORICAL_FEATURES):
    ax = axes.flat[i]
    fraud_rate_by_cat = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    bars = ax.bar(fraud_rate_by_cat.index, fraud_rate_by_cat.values * 100,
                  color="steelblue")
    ax.set_title(f"Fraud Rate by {col}")
    ax.set_ylabel("Fraud Rate (%)")
    ax.tick_params(axis="x", rotation=30)

# Hide unused subplot
axes.flat[-1].set_visible(False)

fig.suptitle("Fraud Rate by Categorical Feature", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/categorical_fraud_rates.png", dpi=150)
plt.show()

## 5. Numerical Feature Distributions

In [ ]:
numerical_cols = [c for c in df.columns if c not in CATEGORICAL_FEATURES + [TARGET, "month"]]
print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")

fraud_df = df[df[TARGET] == 1]
legit_df = df[df[TARGET] == 0]

# Compute point-biserial correlation with target for sorting
corr_with_target = df[numerical_cols + [TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print("\nTop 10 numerical features by |correlation| with fraud_bool:")
print(corr_with_target.head(10).to_string())

In [ ]:
# Plot top 12 most discriminative numerical features
top_num = corr_with_target.head(12).index.tolist()
n_cols = 4
n_rows = (len(top_num) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
for i, col in enumerate(top_num):
    ax = axes.flat[i]
    r = df[col].corr(df[TARGET])
    sns.kdeplot(legit_df[col], ax=ax, color="steelblue", label="Legit", fill=True, alpha=0.3)
    sns.kdeplot(fraud_df[col], ax=ax, color="tomato", label="Fraud", fill=True, alpha=0.5)
    ax.set_title(f"{col}  (r={r:.3f})", fontsize=9)
    ax.set_xlabel("")
    if i == 0:
        ax.legend(fontsize=8)

for j in range(len(top_num), len(axes.flat)):
    axes.flat[j].set_visible(False)

fig.suptitle("Top Discriminative Numerical Features: Fraud vs Legitimate", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/feature_distributions.png", dpi=120)
plt.show()

## 6. Correlation Matrix

In [ ]:
# Correlation among numerical features + target
corr_cols = numerical_cols + [TARGET]
corr = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(
    corr, mask=mask, cmap="RdBu_r", center=0,
    vmin=-1, vmax=1, linewidths=0.3, ax=ax
)
ax.set_title("Numerical Feature Correlation Matrix", fontsize=14)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/correlation_matrix.png", dpi=150)
plt.show()

## 7. Outlier Analysis — Top Discriminative Features

In [ ]:
top8 = corr_with_target.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(top8):
    ax = axes.flat[i]
    data_to_plot = [legit_df[col].dropna(), fraud_df[col].dropna()]
    bp = ax.boxplot(data_to_plot, labels=["Legit", "Fraud"], patch_artist=True)
    bp["boxes"][0].set_facecolor("steelblue")
    bp["boxes"][0].set_alpha(0.6)
    bp["boxes"][1].set_facecolor("tomato")
    bp["boxes"][1].set_alpha(0.6)
    ax.set_title(col, fontsize=10)

fig.suptitle("Boxplots: Top Discriminative Features", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/outlier_analysis.png", dpi=150)
plt.show()

## 8. Summary of EDA Findings

| Finding | Implication for Modeling |
|---|---|
| Class imbalance (~1% fraud) | Use AUPRC as primary metric; use `class_weight='balanced'` or `scale_pos_weight` |
| Temporal structure (8 months) | Use temporal train/val/test split — no random shuffling across time |
| 5 categorical features | Label-encode before training; fit encoders on train only |
| Numerical features vary in scale | StandardScaler on numerical features (needed for logistic regression) |
| Fraud rate may drift over months | Monitor model performance over time in production |
| No missing values | No imputation needed |
| Baseline AUPRC ≈ fraud prevalence | Any useful model should significantly exceed baseline |

**Next steps:** Run `python DATA/preprocess.py` then `python MODELS/train_models.py`.